# Chapter 1 — Defining "good" before writing code

Every number in Chapter 1, recomputed from `evalkit/basics.py`: how many outputs you must read to see a
failure mode, the Wald and Wilson intervals for small counts, the worked example's simulated reading of 100
support conversations, and the exercises' simulations. Every cell **asserts** what it checks.

In [ ]:
import math, os, sys, random
sys.path.insert(0, os.path.abspath(".."))      # run from notebooks/
sys.path.insert(0, os.path.abspath("."))       # or from the repo root
import numpy as np
from evalkit import basics as B
R1 = lambda x: round(x * 100, 1)

## 1.3 — reads needed to see a failure mode: n ≥ ln(1 − c) / ln(1 − p)

In [ ]:
for p in (.01, .02, .03, .05, .10, .20):
    print(f"p={p:5.0%}  reads for 95%: {B.reads_to_see(p):4d}   P(seen) in 100: {B.prob_seen(p, 100):6.1%}   in 30: {B.prob_seen(p, 30):6.1%}")
assert [B.reads_to_see(p) for p in (.01, .02, .03, .05, .10, .20)] == [299, 149, 99, 59, 29, 14]
assert [R1(B.prob_seen(p, 30)) for p in (.01, .02, .03, .05, .10)] == [26.0, 45.5, 59.9, 78.5, 95.8]
# the same number by brute force: simulate 20,000 readings of 100 outputs at p = 3%
rng = np.random.default_rng(0)
sim = (rng.random((20_000, 100)) < 0.03).any(axis=1).mean()
print("simulated P(seen) for p=3%, n=100:", sim)
assert abs(sim - B.prob_seen(0.03, 100)) < 0.005

## 1.3 — Wald against Wilson at small counts

In [ ]:
for k in (0, 1, 3, 5, 12):
    print(k, "Wald", tuple(R1(x) for x in B.wald(k, 100)), " Wilson", tuple(R1(x) for x in B.wilson(k, 100)))
assert tuple(R1(x) for x in B.wald(1, 100)) == (-1.0, 3.0)       # a negative rate
assert tuple(R1(x) for x in B.wald(0, 100)) == (0.0, 0.0)        # certainty from nothing
assert tuple(R1(x) for x in B.wilson(5, 100)) == (2.2, 11.2)
z = B.z_for(); assert round(z * z / (100 + z * z) * 100, 1) == 3.7

## 1.5 — the worked example: 100 support conversations, four failure modes

In [ ]:
RATES = {"wrong policy": 0.06, "invented order status": 0.03, "missed escalation": 0.02, "tone": 0.08}
seen = B.simulate_failure_reading(RATES, 100, seed=1)
for k, v in seen.items():
    lo, hi = B.wilson(v, 100)
    print(f"{k:22s} true {RATES[k]:4.0%}   seen {v:2d}/100   Wilson [{lo:.1%}, {hi:.1%}]")
assert seen == {"wrong policy": 9, "invented order status": 7, "missed escalation": 0, "tone": 13}
assert B.wilson(7, 100)[0] > 0.03          # the one interval that misses its true rate
assert round(0.98 ** 100, 3) == 0.133      # chance the 2% escalation failure is invisible

## Exercises 1.2–1.5

In [ ]:
assert B.reads_to_see(0.005) == 598 and B.reads_to_see(0.005, 0.80) == 322
assert round(1 - 0.05 ** (1 / 200), 4) == 0.0149
cov_wald, cov_wilson = B.interval_coverage(0.02, 100, "wald"), B.interval_coverage(0.02, 100, "wilson")
print("coverage at p=2%, n=100:  Wald", cov_wald, "  Wilson", cov_wilson)
assert (cov_wald, cov_wilson) == (0.8645, 0.9447)
runs = B.reading_runs(RATES, 100)
miss = sum(r["missed escalation"] == 0 for r in runs) / 1000
inv = sum(r["invented order status"] > r["wrong policy"] for r in runs) / 1000
tie = sum(r["invented order status"] == r["wrong policy"] for r in runs) / 1000
print("escalation invisible:", miss, "  3% mode ranked above 6% mode:", inv, "  tied:", tie)
assert (miss, inv, tie) == (0.135, 0.120, 0.083)
item = 1200 * 3e-6 + 300 * 15e-6 + 2000 * 3e-6 + 50 * 15e-6
print(f"cost per item ${item:.4f}, per 1,500-item run ${item * 1500:.2f}")
assert round(item * 1500) == 22